# Notebook BERTopic Vanilla 

_A.Morin, E.Schultz - Atelier Méthodes - 7 Mai 2026_

Ce notebook explore chaque composante de la pipeline BERTopic et illustre l'impact des principaux hyperparamètres.

## Télécharger les librairies importantes

Pour les configurations locales, attention à bien gérer vos espaces virtuels (si pas déjà fait)

In [ ]:
# Retirer les # avant de lancer
# !curl https://raw.githubusercontent.com/emilienschultz/slides-bertopic/refs/heads/main/material/requirements.txt > requirements.txt
# !pip install -r requirements.txt

In [ ]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from umap import UMAP 
from hdbscan import HDBSCAN 
from datasets import load_from_disk
from sklearn.feature_extraction.text import CountVectorizer 
from stopwordsiso import stopwords

On selectionne tout le jeu de données désormais : 6500 documents.

In [ ]:
df = pd.read_csv("./theses-soutenues-curated-stratified.csv")
docs = df["resumes.fr"].to_list()
df.head(5)

## Observer l'impact du changement de modèle de plongement

In [ ]:
topic_model = BERTopic(
    language="french", 
    embedding_model = "Alibaba-NLP/gte-multilingual-base",
)
topic_model.fit(documents=docs)

Cette cellule est très longue à faire tourner. Pour éviter l'attente on peut pré-calculer les embeddings et les renseigner pour éviter les calculs de plongements.

[Instructions pour précalculer les embeddings](https://css-polytechnique.github.io/css-ipp-materials/pages/bertopic-tutorial.html#precompute-your-embeddings).

In [ ]:
ds = load_from_disk(f"./embeddings/gte-multilingual-base-fr-SBERT")
docs = np.array(ds[f"resumes.fr"]) # 6500 rows
embeddings = np.array(ds["embedding"])			 # Shape : 6500 x 768

topic_model = BERTopic(
    language="french", 
    embedding_model = "Alibaba-NLP/gte-multilingual-base", # <- Inutile
)
topic_model.fit(
    documents=docs,
    embeddings=embeddings # renseigne les plongements ici
)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.visualize_documents(
    docs = docs,
    hide_annotations = True,
    embeddings=embeddings # Renseigner les plongements ici
)

:::{.callout-warning}
On continue l'exploration avec ces embeddings: `"./embeddings/gte-multilingual-base-fr-SBERT"`
:::

## Observer l'impact de UMAP

In [ ]:
umap_model = UMAP(
    n_neighbors = 50,  # default : 15
    n_components = 3,  # default : 5
    # Default parameters
    metric = "cosine",
    min_dist=0.0,
    low_memory = False, 
    # random_state=42 <--- Pour retirer l'aléatoire
)


topic_model = BERTopic(
    language="french", 
    embedding_model = "Alibaba-NLP/gte-multilingual-base", # <- Inutile
)
topic_model.fit(
    documents=docs,
    embeddings=embeddings # renseigne les plongements ici
)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.visualize_documents(
    docs = docs,
    hide_annotations = True,
    embeddings=embeddings # Renseigner les plongements ici
)

In [ ]:
topic_model.visualize_hierarchy()

### Observer l'impact de UMAP sur les embeddings

In [ ]:
umap_model_for_visualisation = UMAP(
    n_neighbors = 50,  # default : 15 <-- changer valeur
    n_components = 2,  # Doit forcer à deux pour pouvoir les visualiser
    # Default parameters
    metric = "cosine",
    min_dist=0.0,
    low_memory = False 
    # random_state=42 <--- Pour retirer l'aléatoire
) 

reduced_embeddings = umap_model_for_visualisation.fit_transform(embeddings)

topic_model.visualize_documents(
    docs = docs,
    reduced_embeddings = reduced_embeddings,
    hide_annotations = True, 
    hide_document_hover = True,
    height = 600, 
    width = 1000
)

## Observer l'impact de HDBSCAN

In [ ]:
hdbscan_model = HDBSCAN(
    min_cluster_size=50,  # <--- changer la valeur
    # Default parameters 
    prediction_data=True
)

topic_model = BERTopic(language = "french", hdbscan_model=hdbscan_model)
topic_model.fit(documents=docs, embeddings=embeddings)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.visualize_documents(
    docs = docs,
    hide_annotations = True,
    embeddings=embeddings # Renseigner les plongements ici
)

In [ ]:
topic_model.visualize_hierarchy()

## Observer l'impact de CountVectorizer

In [ ]:
vectorizer_model = CountVectorizer(stop_words = list(stopwords("fr")))

topic_model = BERTopic(language = "french", vectorizer_model = vectorizer_model)
topic_model.fit(documents=docs, embeddings=embeddings)

In [ ]:
topic_model.get_topic_info()

## Exercice: Implémenter une pipeline complète en modifiant chaque élément si besoin

In [ ]:
# à toi de jouer! 